In [ ]:
import os
import json
from dotenv import load_dotenv
from neo4j import GraphDatabase
load_dotenv()


In [ ]:
import json
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

NEO4J_URI = os.environ["NEO4J_LOKAL_URI"]
NEO4J_USER = os.environ["NEO4J_LOKAL_USER"]
NEO4J_PASSWORD = os.environ["NEO4J_LOKAL_PASSWORD"]

JSON_PATH = "tematik_.json"

gagal_ayat = []
gagal_format = []


def merge_tematik(tx, nama):
    tx.run(
        "MERGE (:Tematik {nama: $nama})",
        nama=nama
    )


def merge_sub_tema(tx, parent, child):
    tx.run(
        """
        MATCH (p:Tematik {nama: $parent})
        MATCH (c:Tematik {nama: $child})
        MERGE (p)-[:SUB_TEMA]->(c)
        """,
        parent=parent,
        child=child
    )


def merge_ayat_relation(tx, tema, ayat_id):
    result = tx.run(
        """
        MATCH (t:Tematik {nama: $tema})
        MATCH (a:Ayat {id: $ayat_id})
        MERGE (t)-[:TERKAIT_AYAT]->(a)
        RETURN a
        """,
        tema=tema,
        ayat_id=ayat_id
    )
    if result.single() is None:
        return False
    return True


def is_ayat_list(value):
    if not isinstance(value, list):
        return False
    if not value:
        return False
    for item in value:
        if not isinstance(item, dict):
            return False
        if set(item.keys()) != {"surah", "ayat"}:
            return False
    return True


def traverse(session, node, path, parent=None):
    if not isinstance(node, dict):
        gagal_format.append({
            "path": " > ".join(path),
            "value": node
        })
        return

    for key, value in node.items():
        current_path = path + [key]

        try:
            session.write_transaction(merge_tematik, key)
            if parent is not None:
                session.write_transaction(merge_sub_tema, parent, key)
        except Exception as e:
            gagal_format.append({
                "path": " > ".join(current_path),
                "value": str(e)
            })
            continue

        if isinstance(value, dict):
            traverse(session, value, current_path, key)

        elif isinstance(value, list):
            for item in value:
                if isinstance(item, dict) and set(item.keys()) == {"surah", "ayat"}:
                    ayat_id = f"{item['surah']}:{item['ayat']}"
                    ok = session.write_transaction(
                        merge_ayat_relation, key, ayat_id
                    )
                    if not ok:
                        gagal_ayat.append({
                            "path": " > ".join(current_path),
                            "value": item
                        })

                elif isinstance(item, dict):
                    traverse(session, item, current_path, key)

                else:
                    gagal_format.append({
                        "path": " > ".join(current_path),
                        "value": item
                    })
        else:
            gagal_format.append({
                "path": " > ".join(current_path),
                "value": value
            })


def main():
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD)
    )

    with driver.session() as session:
        traverse(session, data, [])

    driver.close()

    print("=== LAPORAN ===")
    print(f"Gagal Ayat   : {len(gagal_ayat)}")
    print(f"Gagal Format : {len(gagal_format)}")

    if gagal_ayat:
        print("\n-- Detail Gagal Ayat --")
        for e in gagal_ayat:
            print(e)

    if gagal_format:
        print("\n-- Detail Gagal Format --")
        for e in gagal_format:
            print(e)


if __name__ == "__main__":
    main()


In [ ]:
import json
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

# ================================
# KONFIGURASI KONEKSI NEO4J
# ================================
neo4j_url = os.environ["NEO4J_LOKAL_URI"]
neo4j_user = os.environ["NEO4J_LOKAL_USER"]
neo4j_password = os.environ["NEO4J_LOKAL_PASSWORD"]

driver = GraphDatabase.driver(
    neo4j_url,
    auth=(neo4j_user, neo4j_password)
)

session = driver.session()

# ================================
# LOAD FILE JSON
# ================================
with open("READY/NODE_SURAH.json", encoding="utf-8") as f:
    surah_data = json.load(f)

with open("READY/NODE_AYAT.json", encoding="utf-8") as f:
    ayat_data = json.load(f)

with open("READY/NODE_ARTI_NAMA.json", encoding="utf-8") as f:
    arti_nama_data = json.load(f)

with open("READY/NODE_TEMPAT.json", encoding="utf-8") as f:
    tempat_data = json.load(f)

# ================================
# INSERT NODE SURAH
# ================================
for s in surah_data:
    session.run(
        """
        MERGE (s:Surah {id: toInteger($id)})
        SET
            s.nama_arab = $nama_arab,
            s.nama_latin = $nama_latin,
            s.total_ayat = toInteger($total_ayat)
        """,
        id=s["id"],
        nama_arab=s["name"],
        nama_latin=s["nama_latin"],
        total_ayat=s["total_ayat"]
    )

# ================================
# INSERT NODE AYAT + RELATIONSHIP
# ================================
for a in ayat_data:
    session.run(
        """
        MATCH (s:Surah {id: toInteger($id_surah)})
        MERGE (a:Ayat {id: $id})
        SET
            a.ayat = toInteger($ayat),
            a.ayat_arab = $ayat_arab,
            a.ayat_indonesia = $ayat_indonesia,
            a.ayat_inggris = $ayat_inggris
        MERGE (s)-[:MEMILIKI_AYAT]->(a)
        """,
        id_surah=a["id_surah"],
        id=a["surah:ayat"],
        ayat=a["ayat"],
        ayat_arab=a["ayat_arab"],
        ayat_indonesia=a["ayat_bahasa_indonesia"],
        ayat_inggris=a["ayat_bahasa_inggris"]
    )

# ================================
# INSERT NODE ARTI NAMA + RELATIONSHIP
# ================================
for an in arti_nama_data:
    session.run(
        """
        MATCH (s:Surah {id: toInteger($id_surah)})
        MERGE (a:ArtiNama {arti: $arti})
        MERGE (s)-[:MEMILIKI_ARTI]->(a)
        """,
        id_surah=an["id_surah"],
        arti=an["arti_nama"]
    )

# ================================
# INSERT NODE TEMPAT + RELATIONSHIP
# ================================
for t in tempat_data:
    session.run(
        """
        MATCH (s:Surah {id: toInteger($id_surah)})
        MERGE (t:Tempat {lokasi: $lokasi})
        MERGE (s)-[:DITURUNKAN_DI]->(t)
        """,
        id_surah=t["id_surah"],
        lokasi=t["lokasi"]
    )

# ================================
# CLEANUP
# ================================
session.close()
driver.close()

print("Graph berhasil dibangun di Neo4j")


In [ ]:
import json
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

NEO4J_URI = os.environ["NEO4J_LOKAL_URI"]
NEO4J_USER = os.environ["NEO4J_LOKAL_USER"]
NEO4J_PASSWORD = os.environ["NEO4J_LOKAL_PASSWORD"]

gagal_ayat = []
gagal_format = []

def is_ayat(obj):
    return (
        isinstance(obj, dict)
        and "surah" in obj
        and "ayat" in obj
        and isinstance(obj["surah"], int)
        and isinstance(obj["ayat"], int)
    )

def link_parent_child(tx, parent_name, child_name):
    tx.run(
        """
        MERGE (p:Tematik {nama: $parent})
        MERGE (c:Tematik {nama: $child})
        MERGE (p)-[:SUB_TEMA]->(c)
        """,
        parent=parent_name,
        child=child_name
    )

def link_leaf_to_ayat(tx, tema_name, ayat_id):
    result = tx.run(
        """
        MATCH (a:Ayat {id: $ayat_id})
        RETURN a
        """,
        ayat_id=ayat_id
    )
    if result.single() is None:
        gagal_ayat.append(f"{tema_name} -> {ayat_id}")
        return

    tx.run(
        """
        MATCH (t:Tematik {nama: $tema})
        MATCH (a:Ayat {id: $ayat_id})
        MERGE (t)-[:TERKAIT_AYAT]->(a)
        """,
        tema=tema_name,
        ayat_id=ayat_id
    )

def handle_list(tx, lst, parent_name, path):
    ayat_items = []
    nested_items = []

    for item in lst:
        if is_ayat(item):
            ayat_items.append(item)
        elif isinstance(item, dict):
            nested_items.append(item)
        else:
            gagal_format.append(" > ".join(path))
            return

    if ayat_items:
        for ay in ayat_items:
            ayat_id = f"{ay['surah']}:{ay['ayat']}"
            link_leaf_to_ayat(tx, parent_name, ayat_id)

    for obj in nested_items:
        for key, value in obj.items():
            insert_tema(tx, key, value, parent_name, path + [key])

def insert_tema(tx, tema_name, data, parent_name=None, path=None):
    if path is None:
        path = [tema_name]

    tx.run(
        """
        MERGE (:Tematik {nama: $nama})
        """,
        nama=tema_name
    )

    if parent_name is not None:
        link_parent_child(tx, parent_name, tema_name)

    if isinstance(data, dict):
        for key, value in data.items():
            insert_tema(tx, key, value, tema_name, path + [key])

    elif isinstance(data, list):
        handle_list(tx, data, tema_name, path)

    else:
        gagal_format.append(" > ".join(path))

def main():
    with open("tematik_.json", "r", encoding="utf-8") as f:
        data = json.load(f)

    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD)
    )

    with driver.session() as session:
        for root_key, root_val in data.items():
            session.execute_write(insert_tema, root_key, root_val)

    driver.close()

    print("=== LAPORAN GAGAL AYAT ===")
    for g in gagal_ayat:
        print(g)

    print("\n=== LAPORAN GAGAL FORMAT ===")
    for g in gagal_format:
        print(g)

if __name__ == "__main__":
    main()


In [ ]:
import os
import json

# Path input
SURAH_PATH = "RAW/id/surah_id.json"
AYAT_ID_PATH = "RAW/id/ayat_id.json"
AYAT_EN_PATH = "RAW/en/ayat_en.json"

# Path output
OUTPUT_DIR = "TESTING"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Load JSON files
with open(SURAH_PATH, "r", encoding="utf-8") as f:
    surah_data = json.load(f)

with open(AYAT_ID_PATH, "r", encoding="utf-8") as f:
    ayat_id_data = json.load(f)

with open(AYAT_EN_PATH, "r", encoding="utf-8") as f:
    ayat_en_data = json.load(f)

# Prepare lookup for English verses
en_lookup = {}
for surah in ayat_en_data:
    surah_id = surah.get("id")
    verses = surah.get("verses", [])
    for v in verses:
        key = (surah_id, v.get("id"))
        en_lookup[key] = v.get("translation", "")

NODE_SURAH = []
NODE_TEMPAT = []
NODE_ARTI_NAMA = []
NODE_AYAT = []

# Process surah metadata
for surah in surah_data:
    sid = surah.get("id")

    NODE_SURAH.append({
        "id": str(sid),
        "name": surah.get("name", ""),
        "nama_latin": surah.get("transliteration", ""),
        "total_ayat": str(surah.get("total_verses", ""))
    })

    lokasi = "makkah"
    if surah.get("type") != "meccan":
        lokasi = "madinah"

    NODE_TEMPAT.append({
        "id_surah": str(sid),
        "lokasi": lokasi
    })

    NODE_ARTI_NAMA.append({
        "id_surah": str(sid),
        "arti_nama": surah.get("translation", "")
    })

# Process ayat
for surah in ayat_id_data:
    surah_id = surah.get("id")
    verses = surah.get("verses", [])

    for v in verses:
        ayat_id = v.get("id")
        ayat_en = ""
        key = (surah_id, ayat_id)
        if key in en_lookup:
            ayat_en = en_lookup[key]

        NODE_AYAT.append({
            "id_surah": surah_id,
            "ayat": ayat_id,
            "surah:ayat": f"{surah_id}:{ayat_id}",
            "ayat_arab": v.get("text", ""),
            "ayat_bahasa_indonesia": v.get("translation", ""),
            "ayat_bahasa_inggris": ayat_en
        })

# Save outputs
with open(os.path.join(OUTPUT_DIR, "NODE_SURAH.json"), "w", encoding="utf-8") as f:
    json.dump(NODE_SURAH, f, ensure_ascii=False, indent=2)

with open(os.path.join(OUTPUT_DIR, "NODE_TEMPAT.json"), "w", encoding="utf-8") as f:
    json.dump(NODE_TEMPAT, f, ensure_ascii=False, indent=2)

with open(os.path.join(OUTPUT_DIR, "NODE_ARTI_NAMA.json"), "w", encoding="utf-8") as f:
    json.dump(NODE_ARTI_NAMA, f, ensure_ascii=False, indent=2)

with open(os.path.join(OUTPUT_DIR, "NODE_AYAT.json"), "w", encoding="utf-8") as f:
    json.dump(NODE_AYAT, f, ensure_ascii=False, indent=2)

print("SELESAI")


In [ ]:
import json
import os

with open("RAW/id/ayat_id.json", "r", encoding="utf-8") as f:
    ayat_id = json.load(f)   

with open("RAW/en/ayat_en.json", "r", encoding="utf-8") as f:
    ayat_en = json.load(f)  

with open("RAW/id/surah_id.json", "r", encoding="utf-8") as f:
    surah_data = json.load(f) 

NODE_SURAH = []
NODE_AYAT = []
NODE_TEMPAT = []
NODE_ARTI_NAMA = []

for surah in surah_data:
    NODE_SURAH.append({
        "id": str(surah["id"]),
        "name": surah["name"],
        "nama_latin": surah["transliteration"],
        "total_ayat": str(surah["total_verses"])
    })

    NODE_TEMPAT.append({
        "id_surah": str(surah["id"]),
        "lokasi": "makkah" if surah["type"] == "meccan" else "madinah"
    })

    NODE_ARTI_NAMA.append({
        "id_surah": str(surah["id"]),
        "arti_nama": surah["translation"]
    })

english_map = {}

for surah in ayat_en:
    surah_id = surah["id"]
    for verse in surah["verses"]:
        english_map[(surah_id, verse["id"])] = verse["translation"]

for surah in ayat_id:
    surah_id = surah["id"]

    for verse in surah["verses"]:
        verse_id = verse["id"]

        en_text = english_map.get((surah_id, verse_id), "")

        NODE_AYAT.append({
            "id_surah": surah_id,
            "ayat": verse_id,
            "surah:ayat": f"{surah_id}:{verse_id}",
            "ayat_arab": verse["text"],
            "ayat_bahasa_indonesia": verse["translation"],
            "ayat_bahasa_inggris": en_text
        })


os.makedirs("sig", exist_ok=True)

def save_json(filename, data):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

save_json("READY/NODE_SURAH.json", NODE_SURAH)
save_json("READY/NODE_AYAT.json", NODE_AYAT)
save_json("READY/NODE_TEMPAT.json", NODE_TEMPAT)
save_json("READY/NODE_ARTI_NAMA.json", NODE_ARTI_NAMA)

print("SELESAI")


# SESUAIKAN URL, USER, DAN PASSWORD

In [10]:
import os

from dotenv import load_dotenv

load_dotenv()

neo4j_url = os.environ["NEO4J_LOKAL_URI"]
neo4j_user = os.environ["NEO4J_LOKAL_USER"]
neo4j_password = os.environ["NEO4J_LOKAL_PASSWORD"]

# CEK KONEKSI

In [ ]:
from neo4j import GraphDatabase

AUTH = (neo4j_user, neo4j_password)

with GraphDatabase.driver(neo4j_url, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Koneksi berhasil ke database")


In [ ]:
from neo4j import GraphDatabase

def inspect_graph(uri, user, password):
    driver = GraphDatabase.driver(neo4j_url, auth=AUTH)
    try:
        with driver.session() as s:
            total_nodes = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
            total_rels  = s.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]

            per_label = s.run("""
                MATCH (n)
                UNWIND labels(n) AS label
                RETURN label, count(*) AS c
                ORDER BY c DESC
            """).values("label", "c")

            per_type = s.run("""
                MATCH ()-[r]->()
                RETURN type(r) AS type, count(*) AS c
                ORDER BY c DESC
            """).values("type", "c")

            return {
                "total_nodes": total_nodes,
                "total_relationships": total_rels,
                "nodes_per_label": per_label,        
                "relationships_per_type": per_type  
            }
    finally:
        driver.close()

# Pakai:
stats = inspect_graph(neo4j_url, neo4j_user, neo4j_password)
print("URL:", neo4j_url)
print("Total nodes:", stats["total_nodes"])
print("Total relationships:", stats["total_relationships"])
print("Nodes per label:", stats["nodes_per_label"])
print("Relationships per type:", stats["relationships_per_type"])


# MEMBUAT NODE SURAH, AYAT, TEMPAT, ARTINAMA

In [13]:
driver = GraphDatabase.driver(neo4j_url, auth=(neo4j_user, neo4j_password))
session = driver.session()

In [ ]:
with open("READY/NODE_SURAH.json", encoding="utf-8") as f:
    surah_data = json.load(f)

with open("READY/NODE_AYAT.json", encoding="utf-8") as f:
    ayat_data = json.load(f)

with open("READY/NODE_ARTI_NAMA.json", encoding="utf-8") as f:
    arti_nama_data = json.load(f)

with open("READY/NODE_TEMPAT.json", encoding="utf-8") as f:
    tempat_data = json.load(f)
for item in surah_data:
    session.run(
        """
        MERGE (s:Surah {id: $id})
        SET s.nama_arab = $name,
            s.nama_latin = $nama_latin,
            s.total_ayat = toInteger($total_ayat)
        """,
        id=str(item["id"]),
        name=item["name"],
        nama_latin=item["nama_latin"],
        total_ayat=item["total_ayat"]
    )

# === BUAT NODE AYAT dan RELASI KE SURAH ===
for item in ayat_data:
    id_surah = str(item["id_surah"])
    surah_ayat = f"{id_surah}:{item['ayat']}"

    session.run(
        """
        MERGE (a:Ayat {id: $surah_ayat})
        SET a.ayat = toInteger($ayat),
            a.ayat_arab = $ayat_arab,
            a.ayat_indonesia = $ayat_bahasa_indonesia,
            a.ayat_inggris = $ayat_bahasa_inggris
            
        WITH a
        MATCH (s:Surah {id: $id_surah})
        MERGE (s)-[:MEMILIKI_AYAT]->(a)
        """,
        id_surah=id_surah,
        ayat=item["ayat"],
        surah_ayat=surah_ayat,
        ayat_arab=item["ayat_arab"],
        ayat_bahasa_indonesia=item["ayat_bahasa_indonesia"],
        ayat_bahasa_inggris=item["ayat_bahasa_inggris"]
        
    )

# === BUAT NODE ARTI NAMA dan RELASI KE SURAH ===
for item in arti_nama_data:
    session.run(
        """
        MERGE (a:ArtiNama {arti: $arti_nama})
        WITH a
        MATCH (s:Surah {id: $id_surah})
        MERGE (s)-[:MEMILIKI_ARTI]->(a)
        """,
        id_surah=str(item["id_surah"]),
        arti_nama=item["arti_nama"]
    )

# === BUAT NODE TEMPAT dan RELASI KE SURAH ===
for item in tempat_data:
    session.run(
        """
        MERGE (t:Tempat {lokasi: $lokasi})
        WITH t
        MATCH (s:Surah {id: $id_surah})
        MERGE (s)-[:DITURUNKAN_DI]->(t)
        """,
        id_surah=str(item["id_surah"]),
        lokasi=item["lokasi"]
    )

print("✅ Graph berhasil dibangun di Neo4j")

session.close()
driver.close()

# MEBUAT NODE TEMATIK

In [ ]:
import json
from neo4j import GraphDatabase

NEO4J_URI = neo4j_url
NEO4J_USER = neo4j_user
NEO4J_PASSWORD = neo4j_password

with open("tematik_.json", "r", encoding="utf-8") as f:
    data = json.load(f)

gagal_ayat = []
gagal_format = []

def is_ayat(obj):
    return isinstance(obj, dict) and "surah" in obj and "ayat" in obj

def link_parent_child(tx, parent_name, child_name):
    tx.run("""
        MERGE (p:Tematik {nama:$parent})
        MERGE (c:Tematik {nama:$child})
        MERGE (p)-[:SUB_TEMA]->(c)
    """, parent=parent_name, child=child_name)

def link_leaf_to_ayat(tx, leaf_name, ayat_id):
    # hanya buat relasi jika Ayat ada
    res = tx.run("MATCH (a:Ayat {id:$id}) RETURN a", id=ayat_id).single()
    if res:
        tx.run("""
            MERGE (l:Tematik {nama:$leaf})
            WITH l
            MATCH (a:Ayat {id:$id})
            MERGE (l)-[:TERKAIT_AYAT]->(a)
        """, leaf=leaf_name, id=ayat_id)
        return True
    return False

def handle_list(tx, parent_name, current_name, arr, path):
    """arr bisa campuran: [{surah, ayat}, {Subtema: [...]}, ...]"""
    ayat_items = [x for x in arr if is_ayat(x)]
    nested_items = [x for x in arr if isinstance(x, dict) and not is_ayat(x)]

    # Jika ada ayat, current_name merupakan leaf tematik
    if ayat_items:
        link_parent_child(tx, parent_name, current_name)
        for a in ayat_items:
            ayat_id = f"{a['surah']}:{a['ayat']}"
            ok = link_leaf_to_ayat(tx, current_name, ayat_id)
            if not ok:
                gagal_ayat.append({
                    "leaf": current_name,
                    "ayat_id": ayat_id,
                    "path": " > ".join(path + [current_name])
                })

    # Tangani nested object di dalam array sebagai sub-tema berikutnya
    for obj in nested_items:
        for sub_name, sub_val in obj.items():
            link_parent_child(tx, current_name if ayat_items else parent_name, sub_name)
            # Rekursi
            if isinstance(sub_val, list):
                handle_list(tx,
                            current_name if ayat_items else parent_name,
                            sub_name, sub_val, path + [current_name if ayat_items else parent_name])
            elif isinstance(sub_val, dict):
                insert_tema(tx, sub_name, sub_val, path + [sub_name])
            else:
                gagal_format.append({
                    "key": sub_name,
                    "value": sub_val,
                    "path": " > ".join(path + [sub_name])
                })

def insert_tema(tx, parent_name, node, path):
    """
    Aturan:
    - depth 1 & 2: dict berisi sub-objek (tema)
    - depth 3: key -> list yang bisa langsung ayat ATAU nested tema lagi
    - kedalaman bisa lebih (kita rekursi generik)
    """
    if isinstance(node, dict):
        for name, sub in node.items():
            # buat edge parent -> name, lalu lanjutkan
            link_parent_child(tx, parent_name, name)
            if isinstance(sub, dict):
                insert_tema(tx, name, sub, path + [name])
            elif isinstance(sub, list):
                handle_list(tx, parent_name=name, current_name=name, arr=sub, path=path + [name])
            else:
                gagal_format.append({
                    "key": name,
                    "value": sub,
                    "path": " > ".join(path + [name])
                })
    elif isinstance(node, list):
        # Jika fungsi dipanggil dengan list, proses sebagai list campuran di bawah parent_name
        handle_list(tx, parent_name, parent_name, node, path)
    else:
        gagal_format.append({
            "key": parent_name,
            "value": node,
            "path": " > ".join(path)
        })

# Eksekusi
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as session:
    for root_name, child in data.items():
        session.run("MERGE (:Tematik {nama:$name})", name=root_name)
        session.execute_write(insert_tema, root_name, child, [root_name])
driver.close()

# Laporan
if gagal_ayat:
    print("\n❌ Ayat tidak ditemukan:")
    for g in gagal_ayat:
        print(f"  Path: {g['path']} | Ayat ID: {g['ayat_id']}")
if gagal_format:
    print("\n⚠️ Format tidak valid:")
    for g in gagal_format:
        print(f"  Path: {g['path']} | Value: {g['value']}")
print("\n✅ Proses selesai.")


# MENGHAPUS BUG SELF RELATION

In [16]:
from neo4j import GraphDatabase


driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as session:
    session.run("""
        MATCH (t:Tematik)-[r:SUB_TEMA]->(t)
        DELETE r
    """)

driver.close()


In [9]:
from neo4j import GraphDatabase


driver = GraphDatabase.driver(neo4j_url, auth=(neo4j_user, neo4j_password))
with driver.session() as session:
    session.run("""
        MATCH (n)
        DETACH DELETE n
    """)

driver.close()
